# Pair Trading Sandbox

Prototype a simple HYG/JNK return-spread and z-score workflow using ETF Terminal data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from db.connection import get_engine
from stores.market import PriceStore

In [ ]:
DATA_BACKEND = "local"
APP_ENV = "uat"
START_DATE = "2023-01-01"
hedge_window = 60
z_window = 20

In [ ]:
engine = get_engine(data_backend=DATA_BACKEND, app_env=APP_ENV)
price_store = PriceStore(engine)

histories = price_store.get_multi_ticker_price_history(["HYG", "JNK"], start_date=START_DATE)
prices = pd.DataFrame({ticker: frame["adj_close"] for ticker, frame in histories.items()}).dropna()
returns = np.log(prices).diff().dropna()

prices.tail()

In [ ]:
cov_hyg_jnk = returns["HYG"].rolling(hedge_window).cov(returns["JNK"])
var_jnk = returns["JNK"].rolling(hedge_window).var()
beta = cov_hyg_jnk / var_jnk

spread = returns["HYG"] - beta * returns["JNK"]
spread_mean = spread.rolling(z_window).mean()
spread_std = spread.rolling(z_window).std(ddof=0)
zscore = (spread - spread_mean) / spread_std

pair_df = pd.DataFrame({
    "HYG_ret": returns["HYG"],
    "JNK_ret": returns["JNK"],
    "hedge_ratio": beta,
    "spread": spread,
    "zscore": zscore,
}).dropna()

pair_df.tail()

In [ ]:
print("Latest hedge ratio:", round(pair_df["hedge_ratio"].iloc[-1], 4))
print("Latest spread:", round(pair_df["spread"].iloc[-1], 6))
print("Latest z-score:", round(pair_df["zscore"].iloc[-1], 4))

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

pair_df["hedge_ratio"].plot(ax=axes[0], title="Rolling Hedge Ratio")
axes[0].axhline(pair_df["hedge_ratio"].mean(), color="gray", linestyle="--", alpha=0.7)

pair_df["spread"].plot(ax=axes[1], title="Beta-Hedged Return Spread", color="darkorange")
axes[1].axhline(0, color="black", linestyle="--", alpha=0.7)

pair_df["zscore"].plot(ax=axes[2], title="Spread Z-Score", color="teal")
axes[2].axhline(0, color="black", linestyle="--", alpha=0.7)
axes[2].axhline(2, color="red", linestyle="--", alpha=0.7)
axes[2].axhline(-2, color="green", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()